# LF signature — independent single-cell replication (GSE267819)

Tests whether the activated-fibroblast signature from the GSE294458 pilot **reproduces in a second, independent human ligamentum flavum single-cell cohort**: GSE267819 (Ham et al., LF hypertrophy; 3 hypertrophic donors, 10x Genomics). This is the cohort previously thought unavailable — it is public on GEO.

Because the disease signature is a **within-tissue activated-vs-resting fibroblast state contrast**, it does not require matched controls, so 3 hypertrophic samples are sufficient to independently derive the same state signature and compare it to the pilot.

> If the pilot's activated-fibroblast UP program is also up in this cohort's activated-vs-resting contrast, the LF signature is replicated across **two independent single-cell cohorts** (plus the bulk GSE113212 cohort already shown), materially strengthening the preprint. Discovery-stage; not preclinical validation.

In [ ]:
# 1. Install
!pip -q install "scanpy>=1.10" leidenalg python-igraph 2>/dev/null
!pip -q install harmonypy 2>/dev/null
print("done")

In [ ]:
# 2. Imports
import os, glob, tarfile, urllib.request, re
import numpy as np, pandas as pd, scanpy as sc
import scipy.io, scipy.sparse as sp
import anndata as ad
sc.settings.verbosity=1; sc.settings.figdir="figs"
os.makedirs("figs",exist_ok=True); os.makedirs("data",exist_ok=True); os.makedirs("out",exist_ok=True)
print("scanpy", sc.__version__)

## Step 1 — Download GSE267819 from the GEO FTP

In [ ]:
# 3. Fetch supplementary files (User-Agent set; NCBI 403s the default urllib UA)
BASE="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE267nnn/GSE267819/suppl/"
opener=urllib.request.build_opener()
opener.addheaders=[("User-Agent","Mozilla/5.0 (X11; Linux x86_64)")]
urllib.request.install_opener(opener)
def list_ftp_dir(url):
    html=urllib.request.urlopen(url,timeout=60).read().decode("utf-8","ignore")
    return [n for n in re.findall(r'href="([^"?/][^"]*)"',html) if not n.startswith(("http","/"))]
try:
    files=list_ftp_dir(BASE)
except Exception as e:
    print("listing failed (",e,"); falling back to GSE267819_RAW.tar"); files=["GSE267819_RAW.tar"]
print("supplementary files:"); [print("  ",f) for f in files]
for f in files:
    dst=os.path.join("data",f)
    if not os.path.exists(dst):
        print("downloading",f); urllib.request.urlretrieve(BASE+f,dst)
for t in glob.glob("data/*.tar"):
    print("extracting",t)
    with tarfile.open(t) as tf: tf.extractall("data")
print("\nfiles under data/:")
[print("  ",p) for p in sorted(glob.glob("data/**/*",recursive=True))]

## Step 2 — Load into AnnData (GEO-style prefixed triplets; 3 donors)

All samples are hypertrophic LF; each is labelled by its filename prefix so donors can be integrated. If the printed layout differs, adjust the globs.

In [ ]:
# 4. Load prefixed mtx/tsv triplets
def _read_tsv(p): return pd.read_csv(p,header=None,sep="\t",compression="infer")
def load_triplet(mtx):
    d=os.path.dirname(mtx) or "."; base=os.path.basename(mtx)
    prefix=base[:re.search(r"matrix\.mtx",base,re.I).start()]
    sibs=[f for f in os.listdir(d) if f.startswith(prefix)]
    bfile=next(f for f in sibs if "barcode" in f.lower())
    ffile=next(f for f in sibs if ("feature" in f.lower() or "gene" in f.lower()) and "mtx" not in f.lower())
    M=scipy.io.mmread(os.path.join(d,base)).tocsr()
    barc=_read_tsv(os.path.join(d,bfile)).iloc[:,0].astype(str).values
    feats=_read_tsv(os.path.join(d,ffile))
    genes=(feats.iloc[:,1] if feats.shape[1]>1 else feats.iloc[:,0]).astype(str).values
    A=ad.AnnData(X=sp.csr_matrix(M.T)); A.obs_names=barc; A.var_names=genes; A.var_names_make_unique()
    A.obs["sample"]=re.sub(r"[^A-Za-z0-9]+$","",prefix) or prefix
    return A

mtxs=sorted(glob.glob("data/**/*matrix.mtx*",recursive=True))
h5s=glob.glob("data/**/*.h5",recursive=True); h5ads=glob.glob("data/**/*.h5ad",recursive=True)
if h5ads:
    adatas=[]
    for p in h5ads:
        a=sc.read_h5ad(p); a.obs["sample"]=os.path.basename(p); adatas.append(a)
elif mtxs:
    print("MTX triplets:"); [print("  ",m) for m in mtxs]
    adatas=[load_triplet(m) for m in mtxs]
elif h5s:
    adatas=[]
    for p in h5s:
        a=sc.read_10x_h5(p); a.var_names_make_unique(); a.obs["sample"]=os.path.basename(p); adatas.append(a)
else:
    raise FileNotFoundError("No matrices found under data/ - inspect Step 1 listing.")

adata = adatas[0] if len(adatas)==1 else ad.concat(adatas,join="outer",label="batch",index_unique="-")
adata.obs_names_make_unique()
print(adata); print(adata.obs["sample"].value_counts())

## Step 3 — QC (incl. erythrocyte removal)

In [ ]:
# 5. QC
adata.var["mt"]=adata.var_names.str.upper().str.startswith("MT-")
HB=["HBA1","HBA2","HBB","HBD","HBM","HBQ1","HBZ","HBE1"]
adata.var["hb"]=adata.var_names.str.upper().isin(HB)
sc.pp.calculate_qc_metrics(adata,qc_vars=["mt","hb"],percent_top=None,inplace=True)
sc.pp.filter_cells(adata,min_genes=200); sc.pp.filter_genes(adata,min_cells=3)
adata=adata[adata.obs.pct_counts_mt<15].copy()
adata=adata[adata.obs.n_genes_by_counts<6000].copy()
nb=adata.n_obs; adata=adata[adata.obs.pct_counts_hb<20].copy()
print(f"removed {nb-adata.n_obs} RBC-contaminated cells; after QC {adata.shape}")

## Step 4 — Normalize, integrate donors, cluster

In [ ]:
# 6. Normalize + embed (Harmony over donor)
adata.layers["counts"]=adata.X.copy()
sc.pp.normalize_total(adata,target_sum=1e4); sc.pp.log1p(adata); adata.raw=adata
sc.pp.highly_variable_genes(adata,n_top_genes=2000,batch_key="sample")
adata.var.loc[adata.var["mt"]|adata.var["hb"],"highly_variable"]=False
ah=adata[:,adata.var.highly_variable].copy(); sc.pp.scale(ah,max_value=10); sc.tl.pca(ah,n_comps=30)
rep="X_pca"
try:
    sc.external.pp.harmony_integrate(ah,"sample"); rep="X_pca_harmony"; print("Harmony applied")
except Exception as e: print("Harmony skipped:",e)
sc.pp.neighbors(ah,n_neighbors=15,use_rep=rep); sc.tl.leiden(ah,resolution=1.0); sc.tl.umap(ah)
adata.obs["leiden"]=ah.obs["leiden"].values; adata.obsm["X_umap"]=ah.obsm["X_umap"]
print(adata.obs["leiden"].value_counts())

## Step 5 — Compartment annotation

In [ ]:
# 7. Compartments
markers={"Fibroblast":["COL1A1","COL1A2","COL3A1","DCN","LUM","PDGFRA","PDGFRB"],
 "Myofibroblast":["ACTA2","TAGLN","POSTN","FN1","TNC","THBS1","COMP"],
 "Macrophage":["CD68","LYZ","AIF1","CD163","C1QA","C1QB","SPP1","MRC1"],
 "Endothelial":["PECAM1","VWF","CLDN5","CDH5","FLT1"],"Tcell":["PTPRC","CD3D","CD3E","IL7R"],
 "SmoothMuscle":["MYH11","MYL9","DES"],
 "Erythrocyte":["HBB","HBA1","HBA2","ALAS2","GYPA","SLC4A1","AHSP","SLC25A37"]}
for k,gs in markers.items():
    sc.tl.score_genes(adata,[g for g in gs if g in adata.raw.var_names],score_name=f"score_{k}",use_raw=True)
cs=adata.obs.groupby("leiden")[[f"score_{k}" for k in markers]].mean()
adata.obs["compartment"]=adata.obs["leiden"].map(cs.idxmax(axis=1).str.replace("score_","")).astype(str)
print(adata.obs["compartment"].value_counts())

## Step 6 — Activated-vs-resting fibroblast state signature (this cohort)

In [ ]:
# 8. Fibroblast STATE contrast in GSE267819
fib=adata[adata.obs["compartment"].isin(["Fibroblast","Myofibroblast"])].copy()
print("fibroblasts:",fib.shape[0])
fh=fib[:,fib.var_names.isin(adata.var_names[adata.var.highly_variable])].copy()
sc.pp.scale(fh,max_value=10); sc.tl.pca(fh,n_comps=20); sc.pp.neighbors(fh,n_neighbors=15)
sc.tl.leiden(fh,resolution=0.6,key_added="fib_sub"); fib.obs["fib_sub"]=fh.obs["fib_sub"].values
act=[g for g in ["ACTA2","TAGLN","POSTN","FN1","TNC","THBS1","COMP","COL1A1","COL3A1"] if g in fib.raw.var_names]
sc.tl.score_genes(fib,act,score_name="activated_score",use_raw=True)
ss=fib.obs.groupby("fib_sub")["activated_score"].mean().sort_values()
lo,hi=ss.index[0],ss.index[-1]
fib.obs["fib_state"]="intermediate"
fib.obs.loc[fib.obs.fib_sub==lo,"fib_state"]="resting"; fib.obs.loc[fib.obs.fib_sub==hi,"fib_state"]="activated"
st=fib[fib.obs.fib_state.isin(["activated","resting"])].copy()
sc.tl.rank_genes_groups(st,"fib_state",groups=["activated"],reference="resting",method="wilcoxon",use_raw=True)
de=sc.get.rank_genes_groups_df(st,group="activated"); de.to_csv("out/GSE267819_STATE_de.csv",index=False)
print("top UP in activated fibroblasts (this cohort):")
print(de.sort_values("scores",ascending=False).head(20)[["names","logfoldchanges","pvals_adj"]].to_string(index=False))

## Step 7 — Replication test vs the GSE294458 pilot signature

In [ ]:
# 9. Does the pilot activated-fibroblast UP program replicate here?
PILOT_UP=["COL1A2","COL3A1","LUM","ASPN","COL1A1","HTRA1","MFGE8","OGN","TSC22D1","COL5A2",
 "S100A4","ARL6IP5","CD9","DCN","DKK3","CLU","MXRA8","NUPR1","SSPN","SOX5","FN1","POSTN"]
lut=de.set_index("names")
present=[g for g in PILOT_UP if g in lut.index]
sub=lut.loc[present]
up_lfc=sub["logfoldchanges"]
conc=float((up_lfc>0).mean())
# rank enrichment of pilot-UP within this cohort's activated ranking (small rank = up)
ded=de.sort_values("scores",ascending=False).reset_index(drop=True)
ranks=pd.Series(ded.index.values,index=ded["names"])
from scipy import stats
r=ranks.reindex(present).dropna().values
U,p=stats.mannwhitneyu(r,ranks.values,alternative="less")
print(f"pilot-UP genes present: {len(present)}/{len(PILOT_UP)}")
print(f"concordance (frac log2FC>0 in this cohort): {conc:.2f}")
print(f"median activated log2FC of pilot-UP genes: {up_lfc.median():+.3f}")
print(f"rank-enrichment of pilot-UP in this cohort's activated ranking: p = {p:.2e}")
# overlap of top-50 activated genes between cohorts (this cohort vs pilot list)
top50=set(ded.head(50)["names"]); ov=top50 & set(PILOT_UP)
print(f"pilot-UP genes among this cohort's top-50 activated: {sorted(ov)}")
ok=(conc>=0.6 and up_lfc.median()>0 and p<0.05)
print("\nVERDICT:", "REPLICATES across a second independent single-cell cohort." if ok
      else "does not cleanly replicate - inspect per-gene table.")

## Step 8 — Figure

In [ ]:
# 10. Plot pilot-UP genes' activated log2FC in GSE267819
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
d=sub.sort_values("logfoldchanges")
fig,ax=plt.subplots(figsize=(7.5,7),dpi=140)
cols=["#0E8F7E" if (pv<0.05 and lf>0) else "#B9C2CC" for lf,pv in zip(d["logfoldchanges"],d["pvals_adj"])]
ax.barh(range(len(d)),d["logfoldchanges"],color=cols)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index,fontsize=10)
ax.axvline(0,color="#888",lw=1); ax.set_xlabel("activated-vs-resting log2FC in GSE267819")
ax.set_title("Pilot activated-fibroblast signature,\nre-tested in an independent LF single-cell cohort (GSE267819)",fontsize=12,fontweight="bold")
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
ax.tick_params(axis="y",length=0); fig.tight_layout()
fig.savefig("figs/GSE267819_replication.png",dpi=140); print("saved figs/GSE267819_replication.png")

## Interpretation

- A clean replication here means the LF activated-fibroblast signature holds across **two independent single-cell cohorts** (GSE294458 + GSE267819) and one bulk cohort (GSE113212, different platform) — a genuinely well-replicated disease signature, which is a substantial upgrade to the preprint's evidence base.
- The state-contrast design is what makes this possible with 3 control-free hypertrophic samples: we compare activated vs resting fibroblasts within the tissue, not disease vs control.
- If it replicates, update the preprint and repo to report two single-cell cohorts; if not, inspect whether donor/integration or a different fibroblast substructure explains the difference before concluding.